# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution – Exploration with `mlcroissant`

This notebook provides a practical walkthrough for loading and exploring the FAIR\(^2\) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We also import pandas which is used throughout this notebook.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, their `@id` values, and the fields/columns in each. All references should use the `@id`.

**Tip:** The Croissant schema defines each record set and their field and column structure. 

In [ ]:
# List all record sets with their @id, name, and description
recordsets = list(dataset.record_sets())

print("Available record sets:")
for rs in recordsets:
    print(f"- @id: {rs.id}")
    if hasattr(rs, 'name'):
        print(f"  name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  description: {rs.description}")
    print("")
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.id}")
    print("")

Let's quickly view the first few records in each record set using their `@id`.


In [ ]:
for rs in recordsets:
    print(f"First 2 records from record set @id: {rs.id}")
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        print(rec)
        if i > 0:
            break
    print('-' * 60)

## 3. Data Extraction
Load the data for each record set into separate pandas DataFrames. This enables further analysis and visualization. Keys correspond to the record set `@id`s.

In [ ]:
# Prepare to extract all RecordSets using @id
record_set_ids = [rs.id for rs in recordsets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"@id: {record_set_id} -> {len(df)} records, columns: {df.columns.tolist()}")
    print(df.head(2))
    print('-' * 50)

# For further examples, choose the largest record set (by number of rows or by domain knowledge)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nWorking with main record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We perform exploratory steps such as filtering on a numeric field, normalization, and grouping/categorization using the `@id` for all data references.

**Note:** Adjust `numeric_field_id` and `group_field_id` to match column names (i.e., field `@id`), as found above.

In [ ]:
# Select main dataframe
df_main = dataframes.get(main_record_set_id)

# If you know the field names, set them here. For demo, we'll attempt to auto-detect a numeric field.
numeric_field_id = None
if df_main is not None:
    # Heuristically try to find integer/float columns
    for col in df_main.columns:
        if pd.api.types.is_numeric_dtype(df_main[col]):
            numeric_field_id = col
            break

    if not numeric_field_id:
        # If no numeric columns, display available columns
        print("No numeric columns found. Available columns:")
        print(df_main.columns)

if df_main is not None and numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
    # Filtering: choose a basic threshold (for demo let's use median as threshold)
    threshold = df_main[numeric_field_id].median()
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records where {{{numeric_field_id}}} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping: try to select a categorical field
    group_field_id = None
    for col in df_main.columns:
        if col != numeric_field_id and df_main[col].nunique() < len(df_main) / 2:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("Cannot perform numeric filtering/grouping because no suitable numeric field was found.")

## 5. Visualization

Visualize the distribution of the main numeric field and its relationship with the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure main DataFrame and fields are defined
if df_main is not None and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_main[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(f'{numeric_field_id}')
    plt.tight_layout()
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_main)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.tight_layout()
        plt.show()
else:
    print('Insufficient numeric data to plot.')

## 6. Conclusion

- This notebook demonstrated how to load and explore the FAIR$^2$ dataset using `mlcroissant`.
- We reviewed available record sets and fields by their `@id`, extracted data, and performed basic EDA and visualization.
- Further research can involve in-depth domain analysis, advanced statistics, or machine learning on the loaded DataFrames.

For more details about the dataset, schema, and usage, see the [Croissant schema reference](https://github.com/mlcommons/croissant/) and the dataset's metadata.